***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](#)
    * 下一节：[2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)

***


# 第 2 章：干涉测量的数学工具箱<a id='math:sec:intro'></a>


第 1 章介绍了射电干涉测量的科学背景和仪器动机；本章转入描述、采样和反演所需的数学框架。核心对象包括复数与相位、傅里叶表示、离散化、矩阵方程和最小二乘估计。这些对象不是彼此独立的预备知识，而是后续可见度、成像和校准推导的共同语言。

本章按照干涉测量的信息处理链组织。电磁波的振幅与相位由复数表示；两个阵元的相关把相位差写入复可见度；可见度在空间频率平面上采样天空亮度；有限采样、噪声和离散化进一步使成像与校准成为矩阵形式的反问题。傅里叶变换、卷积、相关、采样理论和最小二乘法分别描述这条链中的一个环节。

本章假定读者已经掌握微积分和基础线性代数。内容重点放在第 4 至第 8 章会反复使用的定义、变换规则和数值近似；公式推导保留影响物理解释或计算实现的关键步骤，纯数学证明只在其能够澄清适用条件时给出。


#### 从相位语言到反演语言

第二章的主线可以概括为一条从“表示”走向“反演”的链条。复数首先把振幅和相位放进同一个对象；傅里叶语言把天空方向结构与基线采样联系起来；卷积和相关解释了仪器响应如何改写信号；离散傅里叶变换与采样理论说明连续公式进入计算机后会发生什么；线性代数和最小二乘则把带噪数据、模型参数和求解算法组织成可以实际操作的形式。

![第二章的数学工具链](figures/chapter2_math_toolchain.png)

**图 2.0.1** 第二章的数学工具链。图中的箭头不是严格的逻辑依赖图，而是射电干涉测量中最常见的思维流向：从电场相位出发，经复相关得到可见度，再通过傅里叶、采样和反演语言进入成像与校准。


#### 阅读路径

第二章不要求所有读者按文件编号线性阅读。2.12--2.14 是从主线分出的几何和计算专题；2.15 则直接承接 2.11，是带噪反演主线的一部分。建议按下表选路：

| 路径 | 页面 | 完成标准 |
|:---|:---|:---|
| 核心 | 2.1，2.4--2.11，2.15，2.y.1--2.y.6 | 能固定符号约定，解释有限采样，并用协方差和条件数判断一个反演解 |
| 桥接/复习 | 2.2--2.3，2.12--2.13 | 在 delta、窗函数、傅里叶级数、立体角或球面几何先修不足时按需回查 |
| 选做计算 | 2.14，基 2 FFT 作业，2.y 后续 worked examples | 从定义实现算法，并用数值残差而不是图形相似判断正确性 |

本科高年级课程应完成核心路径；研究生或计算方向课程可加入选做计算路径。桥接页面不是低层次内容，而是为了避免已经具备先修知识的读者被迫重复完整推导。

#### 全书数学约定<a id='math:sec:canonical_conventions'></a>

下表是后续章节引用的标准页。局部推导可以换变量名，但不能静默改变指数符号、共轭次序、复方差或权重含义；与外部软件和文献比较时，应先把对方约定映射到这里。

| 对象 | 本书约定 | 必须同时说明的边界 |
|:---|:---|:---|
| 连续 Fourier 对 | $F(s)=\int f(x)e^{-2\pi ixs}dx$；$f(x)=\int F(s)e^{+2\pi ixs}ds$ | $x$ 与 $s$ 单位互为倒数；二维天空/`uv` 对沿用相同符号 |
| DFT/FFT | $Y_k=\sum_n y_ne^{-2\pi ink/N}$；$y_n=N^{-1}\sum_kY_ke^{+2\pi ink/N}$ | 与 NumPy 默认一致；`fftshift` 只重排索引，不改变变换定义 |
| 数学互相关 | $(f\star g)(\tau)=\int f^*(t)g(t+\tau)dt$ | 工程相关积 $V_{pq}=\langle E_pE_q^*\rangle$ 的共轭次序对应 $(E_q\star E_p)(0)$；积分与时间平均的归一化需另行说明，不能只凭下标猜相位符号 |
| 复统计 | $\boldsymbol C=\mathbb E[(\boldsymbol z-\boldsymbol\mu)(\boldsymbol z-\boldsymbol\mu)^H]$，$\boldsymbol P=\mathbb E[(\boldsymbol z-\boldsymbol\mu)(\boldsymbol z-\boldsymbol\mu)^T]$ | proper 复高斯噪声有 $\boldsymbol P=0$；标量 $\sigma_c^2=\mathbb E|z-\mu|^2$，其实部和虚部方差各为 $\sigma_c^2/2$ |
| 统计权重 | $\boldsymbol r=\boldsymbol d-\boldsymbol m$，$\chi^2=\boldsymbol r^H\boldsymbol C^{-1}\boldsymbol r$，$\boldsymbol W_{\rm stat}=\boldsymbol C^{-1}$ | 独立复样本有 $w_i=1/\sigma_{c,i}^2$；flag 样本应从似然中排除，在对角权重表示中等价于令 $w_i=0$ |
| 成像重加权 | $w_i^{\rm image}=q_iw_i^{\rm stat}$，其中 $q_i$ 可表示密度、Briggs 或 taper 因子 | 它改变 PSF、噪声与尺度响应，不应再解释为原始噪声协方差的逆 |
| 向量与伴随 | 向量默认为列向量，$T$ 为转置，$*$ 为标量共轭，$H$ 为共轭转置 | 对复测量方程，正规方程和协方差必须使用 $H$ |

基线方向、可见度下标、Stokes $V$ 和亮度矩阵归一化还涉及仪器定义，将在第 4 与第 7 章的章首约定表中固定；它们必须与本页的 Fourier 和相关约定显式相容。

#### 本章结构

第 2.1 到 2.2 节先建立最基本的表示语言。复数、相量和一组常见函数原型看似基础，却几乎在全书后续每个技术主题中都会反复出现：平面波、相位差、点源、主波束、有限孔径和采样都要依赖这些概念来表达。

第 2.3 到 2.7 节构成本章的核心主线，即傅里叶语言本身。傅里叶级数从周期信号出发，连续傅里叶变换把这种思想推广到非周期函数，卷积、相关和傅里叶定理则给出“信号经过仪器响应后会怎样改变”的统一描述。后续可见度空间、脏图像、脏波束和孔径合成的很多直觉，都是从这里开始真正稳定下来。

第 2.8 到 2.9 节转向离散化与数值实现。真实观测并不提供连续函数，而只给出有限、离散、带噪的样本；因此 DFT、FFT、采样定理和混叠不是附属技术，而是理解成像限制与数值实现的关键。许多看似“软件细节”的现象，例如谱泄漏、周期延拓、网格化误差和边界效应，本质上都来自这里。

第 2.10、2.11 和 2.15 节把前面的函数语言进一步组织成矩阵、优化与统计语言。这会为第 4 章之后越来越常见的测量方程、正规方程、协方差、参数估计和迭代求解打下基础。射电干涉测量中的许多问题并不是“写出一个漂亮公式”就结束，而是要在不完整、带噪、尺度差异很大的数据中稳定地求解。

第 2.12 到 2.13 节属于几何桥接专题，分别服务于辐射量中的立体角语言和第 3 章的天球几何。第 2.14 节是选做的一维 CLEAN 计算实验，把傅里叶、采样、正规方程和病态反演连接起来。章末练习提供 100 分综合问题集，另保留分段卷积 worked examples 和 FFT 编程作业。

#### 章节导航

1. [2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)
2. [2.2 干涉测量中常见的函数原型](2_2_important_functions.ipynb)
3. [2.3 傅里叶级数：从周期展开到谱表示](2_3_fourier_series.ipynb)
4. [2.4 连续傅里叶变换：从信号表示到可见度语言](2_4_the_fourier_transform.ipynb)
5. [2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)
6. [2.6 互相关、自相关与相似性度量](2_6_cross_correlation_and_auto_correlation.ipynb)
7. [2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
8. [2.8 离散傅里叶变换与 FFT：从解析公式到可计算算法](2_8_the_discrete_fourier_transform.ipynb)
    - [编程作业：实现基 2 FFT](fft_implementation_assignment.ipynb)
9. [2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)
10. [2.10 线性代数：从测量方程到矩阵表示](2_10_linear_algebra.ipynb)
11. [2.11 最小二乘与参数估计](2_11_least_squares.ipynb)
12. [2.12 补充专题：立体角与天球面积元素](2_12_solid_angle.ipynb)
13. [2.13 补充专题：球面三角学](2_13_spherical_trigonometry.ipynb)
14. [2.14 补充专题：一维 CLEAN 的数学演示](2_14_CLEAN_in_1D.ipynb)
15. [2.15 统计、不确定度、正则化与过拟合](2_15_statistics_uncertainty_regularization.ipynb)
16. [2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)
17. [2.y 练习](2_y_exercises.ipynb)


***

下一节：[2.1 复数、相位与相量表示](2_1_complex_numbers.ipynb)
